In [39]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", None)

In [17]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "Extracted_Data"

OUTPUT_DIR = PROJECT_DIR / "processed"

OUTPUT_DIR.mkdir(exist_ok=True)

print(DATA_DIR)

c:\Users\vikhy\Desktop\Nikitha project\Extracted_Data


In [30]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "Extracted_Data"

OUTPUT_DIR = PROJECT_DIR / "processed"

OUTPUT_DIR.mkdir(exist_ok=True)

excel_files = list(DATA_DIR.rglob("*.xlsx"))

print("Excel Files:", len(excel_files))

Excel Files: 24


In [31]:
def extract_metadata(filepath):

    week = filepath.parents[1].name

    site = filepath.parent.name

    latitude = np.nan
    longitude = np.nan

    match = re.search(
        r'([-+]?\d+\.\d+),([-+]?\d+\.\d+)',
        site
    )

    if match:

        latitude = float(match.group(1))
        longitude = float(match.group(2))

        site = re.sub(
            r'([-+]?\d+\.\d+),([-+]?\d+\.\d+)',
            '',
            site
        ).strip()

    return week, site, latitude, longitude

In [33]:
def load_sheet(file, sheet):

    raw = pd.read_excel(
        file,
        sheet_name=sheet,
        header=None
    )

    return raw

In [34]:
def detect_data_start(df):

    for i in range(len(df)):

        first_value = str(df.iloc[i,0])

        if ":" in first_value:

            return i

    return None

In [35]:
def process_sheet(file, sheet):

    raw = load_sheet(file, sheet)

    start_row = detect_data_start(raw)

    if start_row is None:
        return None

    data = raw.iloc[start_row:].copy()

    data.reset_index(drop=True, inplace=True)

    columns = ["Time"]

    for i in range(1, len(data.columns)):
        columns.append(f"Direction_{i}")

    data.columns = columns

    return data

In [26]:
sample = excel_files[0]

excel = pd.ExcelFile(sample)

for sheet in excel.sheet_names:

    if sheet in IGNORE:
        continue

    header = detect_header(sample, sheet)

    print(f"{sheet:25} -> {header}")

Gesamt-Kfz                -> 2
SV >3.5t                  -> 2
LV <3.5t                  -> 2
Pkw                       -> 2
Pkw mit Anhänger          -> 2
Lkw                       -> 2
Lkw mit Anhänger          -> 2
Bus                       -> 2
Lieferwagen               -> 2
Kraftrad                  -> 2
Sattel-Kfz                -> 2


In [36]:
sample = excel_files[0]

xls = pd.ExcelFile(sample)

print(xls.sheet_names)

['Gesamt-Kfz', 'SV >3.5t', 'LV <3.5t', 'Pkw', 'Pkw mit Anhänger', 'Lkw', 'Lkw mit Anhänger', 'Bus', 'Lieferwagen', 'Kraftrad', 'Sattel-Kfz', 'Erklärungen']


In [37]:
test_df = process_sheet(
    sample,
    "Gesamt-Kfz"
)

test_df.head()

,Time,Direction_1,Direction_2,Direction_3,Direction_4,Direction_5,Direction_6,Direction_7,Direction_8,Direction_9,Direction_10,Direction_11,Direction_12,Direction_13,Direction_14,Direction_15,Direction_16,Direction_17,Direction_18,Direction_19,Direction_20,Direction_21,Direction_22
0,00:00:00,5,43,9,17,NaN,NaN,74,0,NaN,NaN,NaN,NaN,NaN,5,52,0,43,0,0,26,0,0
1,00:15:00,4,31,15,14,NaN,NaN,64,0,NaN,NaN,NaN,NaN,NaN,4,46,0,31,0,0,29,0,0
2,00:30:00,5,35,7,23,NaN,NaN,70,0,NaN,NaN,NaN,NaN,NaN,5,42,0,35,0,0,30,0,0
3,00:45:00,4,36,4,15,NaN,NaN,59,0,NaN,NaN,NaN,NaN,NaN,4,40,0,36,0,0,19,0,0
4,01:00:00,6,15,7,15,NaN,NaN,43,0,NaN,NaN,NaN,NaN,NaN,6,22,0,15,0,0,22,0,0


In [40]:
IGNORE = ["Erklärungen"]

master = []

for file in tqdm(excel_files):

    week, site, lat, lon = extract_metadata(file)

    try:
        xls = pd.ExcelFile(file)

    except:
        continue

    for sheet in xls.sheet_names:

        if sheet in IGNORE:
            continue

        try:

            df = process_sheet(
                file,
                sheet
            )

            if df is None:
                continue

            df["Week"] = week
            df["Measurement_Site"] = site
            df["Latitude"] = lat
            df["Longitude"] = lon
            df["Vehicle_Category"] = sheet
            df["Source_File"] = file.name

            master.append(df)

        except Exception as e:

            print(file.name)
            print(sheet)
            print(e)

100%|██████████| 24/24 [00:35<00:00,  1.50s/it]


In [41]:
master_df = pd.concat(
    master,
    ignore_index=True
)

master_df.shape

(193596, 35)

In [42]:
meta_cols = [

    "Week",
    "Measurement_Site",
    "Latitude",
    "Longitude",
    "Vehicle_Category",
    "Source_File",
    "Time"
]

other_cols = [

    c for c in master_df.columns
    if c not in meta_cols
]

master_df = master_df[
    meta_cols + other_cols
]

In [43]:
direction_cols = [

    c for c in master_df.columns

    if c.startswith("Direction_")
]

In [44]:
master_df["Total_Traffic"] = (

    master_df[direction_cols]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .sum(axis=1)
)

In [45]:
master_df.head()

,Week,Measurement_Site,Latitude,Longitude,Vehicle_Category,Source_File,Time,Direction_1,Direction_2,Direction_3,Direction_4,Direction_5,Direction_6,Direction_7,Direction_8,Direction_9,Direction_10,Direction_11,Direction_12,Direction_13,Direction_14,Direction_15,Direction_16,Direction_17,Direction_18,Direction_19,Direction_20,Direction_21,Direction_22,Direction_23,Direction_24,Direction_25,Direction_26,Direction_27,Direction_28,Total_Traffic
0,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:00:00,5,43,9,17,NaN,NaN,74,0,NaN,NaN,NaN,NaN,NaN,5,52,0,43,0,0,26,0,0,NaN,NaN,NaN,NaN,NaN,NaN,274.0
1,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:15:00,4,31,15,14,NaN,NaN,64,0,NaN,NaN,NaN,NaN,NaN,4,46,0,31,0,0,29,0,0,NaN,NaN,NaN,NaN,NaN,NaN,238.0
2,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:30:00,5,35,7,23,NaN,NaN,70,0,NaN,NaN,NaN,NaN,NaN,5,42,0,35,0,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,252.0
3,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:45:00,4,36,4,15,NaN,NaN,59,0,NaN,NaN,NaN,NaN,NaN,4,40,0,36,0,0,19,0,0,NaN,NaN,NaN,NaN,NaN,NaN,217.0
4,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,01:00:00,6,15,7,15,NaN,NaN,43,0,NaN,NaN,NaN,NaN,NaN,6,22,0,15,0,0,22,0,0,NaN,NaN,NaN,NaN,NaN,NaN,151.0


In [51]:

master_df.to_csv(
    OUTPUT_DIR / "master_dataset.csv",
    index=False
)



print("Saved successfully")

Saved successfully


In [50]:
print("Rows:", len(master_df))

print("Sites:",
      master_df["Measurement_Site"].nunique())

print("Vehicle Categories:",
      master_df["Vehicle_Category"].nunique())

print(master_df["Vehicle_Category"].unique())

Rows: 193596
Sites: 11
Vehicle Categories: 15
<StringArray>
[      'Gesamt-Kfz',         'SV >3.5t',         'LV <3.5t',
              'Pkw', 'Pkw mit Anhänger',              'Lkw',
 'Lkw mit Anhänger',              'Bus',      'Lieferwagen',
         'Kraftrad',       'Sattel-Kfz',           'Gesamt',
   'Person Fahrrad',             'Tram',        'Sattelzug']
Length: 15, dtype: str


In [52]:
#verify if all the files are processed
master_df.shape

(193596, 36)

In [53]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 193596 entries, 0 to 193595
Data columns (total 36 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Week              193596 non-null  str    
 1   Measurement_Site  193596 non-null  str    
 2   Latitude          64944 non-null   float64
 3   Longitude         64944 non-null   float64
 4   Vehicle_Category  193596 non-null  str    
 5   Source_File       193596 non-null  str    
 6   Time              191976 non-null  object 
 7   Direction_1       191636 non-null  object 
 8   Direction_2       175404 non-null  object 
 9   Direction_3       79024 non-null   object 
 10  Direction_4       95084 non-null   object 
 11  Direction_5       142690 non-null  object 
 12  Direction_6       110420 non-null  object 
 13  Direction_7       62546 non-null   object 
 14  Direction_8       62106 non-null   object 
 15  Direction_9       15774 non-null   object 
 16  Direction_10      15840 non-nul

In [54]:
master_df["Vehicle_Category"].unique()

<StringArray>
[      'Gesamt-Kfz',         'SV >3.5t',         'LV <3.5t',
              'Pkw', 'Pkw mit Anhänger',              'Lkw',
 'Lkw mit Anhänger',              'Bus',      'Lieferwagen',
         'Kraftrad',       'Sattel-Kfz',           'Gesamt',
   'Person Fahrrad',             'Tram',        'Sattelzug']
Length: 15, dtype: str

In [55]:
master_df["Vehicle_Category"].unique()

<StringArray>
[      'Gesamt-Kfz',         'SV >3.5t',         'LV <3.5t',
              'Pkw', 'Pkw mit Anhänger',              'Lkw',
 'Lkw mit Anhänger',              'Bus',      'Lieferwagen',
         'Kraftrad',       'Sattel-Kfz',           'Gesamt',
   'Person Fahrrad',             'Tram',        'Sattelzug']
Length: 15, dtype: str

In [56]:
master_df.head()

,Week,Measurement_Site,Latitude,Longitude,Vehicle_Category,Source_File,Time,Direction_1,Direction_2,Direction_3,Direction_4,Direction_5,Direction_6,Direction_7,Direction_8,Direction_9,Direction_10,Direction_11,Direction_12,Direction_13,Direction_14,Direction_15,Direction_16,Direction_17,Direction_18,Direction_19,Direction_20,Direction_21,Direction_22,Direction_23,Direction_24,Direction_25,Direction_26,Direction_27,Direction_28,Total_Traffic
0,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:00:00,5,43,9,17,NaN,NaN,74,0,NaN,NaN,NaN,NaN,NaN,5,52,0,43,0,0,26,0,0,NaN,NaN,NaN,NaN,NaN,NaN,274.0
1,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:15:00,4,31,15,14,NaN,NaN,64,0,NaN,NaN,NaN,NaN,NaN,4,46,0,31,0,0,29,0,0,NaN,NaN,NaN,NaN,NaN,NaN,238.0
2,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:30:00,5,35,7,23,NaN,NaN,70,0,NaN,NaN,NaN,NaN,NaN,5,42,0,35,0,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,252.0
3,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,00:45:00,4,36,4,15,NaN,NaN,59,0,NaN,NaN,NaN,NaN,NaN,4,40,0,36,0,0,19,0,0,NaN,NaN,NaN,NaN,NaN,NaN,217.0
4,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,Gesamt-Kfz,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,01:00:00,6,15,7,15,NaN,NaN,43,0,NaN,NaN,NaN,NaN,NaN,6,22,0,15,0,0,22,0,0,NaN,NaN,NaN,NaN,NaN,NaN,151.0
